                    RAG SYSTEM

Question
   │
   ▼
Embedding Model
   │
   ▼
Retriever: Get chucks of ducments suitable to query
   │
   ▼
Top-K Chunks
   │
   ▼
Prompt Builder
   │
   ▼
Generator: Use LLM api  to get answers from the prompt
   │
   ▼
Answer

In [9]:
from src.config import RAW_DOCUMENTS_DIR, TOP_K
from src.loader import load_documents
from src.pipeline import build_knowledge_base
from src.retriever import retrieve
from src.rag import create_rag_prompt

In [12]:
documents = load_documents(
    RAW_DOCUMENTS_DIR
)

print(f"Loaded {len(documents)} documents.")

for document in documents:
    print(document["source"])

Loaded 1 documents.
company_policy.txt


In [13]:
model, index, chunks = build_knowledge_base(
    documents
)

print(f"Created {len(chunks)} chunks.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2324.64it/s]


Created 1 chunks.


In [14]:
print(chunks[0])

{'text': "Company Annual Leave Policy Employees receive 30 days of annual leave per calendar year. Employees should request annual leave through the company's HR system. Unused annual leave may be carried over to the following year, subject to company policy. Employees should normally request leave at least two weeks in advance. Remote Work Policy Employees may work remotely up to three days per week with manager approval. Employees working remotely must remain available during their agreed working hours.", 'source': 'company_policy.txt'}


In [15]:
question = "How many days of annual leave do employees receive?"

print(question)

How many days of annual leave do employees receive?


In [16]:
retrieved_chunks = retrieve(
    query=question,
    model=model,
    index=index,
    chunks=chunks,
    top_k=TOP_K,
)

In [17]:
prompt = create_rag_prompt(
    question=question,
    retrieved_chunks=retrieved_chunks,
)

print(prompt)

You are a corporate knowledge assistant.

Answer the user's question using only the provided context.

If the context does not contain enough information to answer
the question, say that the information was not found in the
provided documents.

Do not invent facts.

Context:
[Source 1: company_policy.txt]
Company Annual Leave Policy Employees receive 30 days of annual leave per calendar year. Employees should request annual leave through the company's HR system. Unused annual leave may be carried over to the following year, subject to company policy. Employees should normally request leave at least two weeks in advance. Remote Work Policy Employees may work remotely up to three days per week with manager approval. Employees working remotely must remain available during their agreed working hours.

Question:
How many days of annual leave do employees receive?

Answer:


In [18]:
print("QUESTION")
print(question)

print("\nRETRIEVED CONTEXT")
for result in retrieved_chunks:
    print(result["text"])

print("\nRAG PROMPT")
print(prompt)

QUESTION
How many days of annual leave do employees receive?

RETRIEVED CONTEXT
Company Annual Leave Policy Employees receive 30 days of annual leave per calendar year. Employees should request annual leave through the company's HR system. Unused annual leave may be carried over to the following year, subject to company policy. Employees should normally request leave at least two weeks in advance. Remote Work Policy Employees may work remotely up to three days per week with manager approval. Employees working remotely must remain available during their agreed working hours.

RAG PROMPT
You are a corporate knowledge assistant.

Answer the user's question using only the provided context.

If the context does not contain enough information to answer
the question, say that the information was not found in the
provided documents.

Do not invent facts.

Context:
[Source 1: company_policy.txt]
Company Annual Leave Policy Employees receive 30 days of annual leave per calendar year. Employees s

In [19]:
from src.generator import generate_answer

answer = generate_answer(prompt)

print(answer)

The RAG prompt has been successfully created. A real language model needs to be connected to generate the final answer.
